# 08 · EDA step 7: build and validate the model's feature set

**Question:** which of the shortlisted features, used **together**, make the best pre-game
win model, and how good is that model on games it never saw?

**Why this step exists.** Steps 5 and 6 scored features one at a time. Features overlap: a
team that outshoots its opponents also tends to outscore them. So a feature that's strong
alone can add almost nothing once a related one is in the model. This step adds features in
small groups, keeps only the ones that improve the combined model, and then measures the
final model once on held-back games.

## Rules (fixed before running)

1. **Model form: regularized logistic regression with a home ice switch.**
   - The score is `b0 × home_ice + b1 × feature1 + ...`, where `home_ice` = 1 for a home game
     and 0 at a neutral site.
   - **Why the switch:** at a neutral site neither team has home ice, so evenly matched teams
     should be 50/50. With a plain intercept, the 53% home edge would also be applied to
     neutral-site games. A separate home-vs-neutral feature can't be learned from 54
     neutral-site games (step 6). The switch learns the home edge from the 903 home games
     and fixes it at zero at neutral sites by rule.
   - **Why no centering:** every feature is home minus away, so 0 means "evenly matched".
     Centering the features would shift part of the home edge into them. They're scaled to
     unit spread only, so the regularization treats them equally.
   - **Why regularized:** with about 800 training games, a small penalty on large
     coefficients guards against overfitting. It's held at scikit-learn's default (C = 1)
     while choosing features, then tuned once at the end (rule 4).
2. **Data and folds:** the same 957 games and `TimeSeriesSplit(5)` as before. **Folds 1–4 for
   choosing, fold 5 untouched** until the final evaluation.
3. **Forward selection, one group at a time, in a fixed order** (step 5's ranking):
   - The model starts with the strongest feature, close-game shot share.
   - Each later group is **kept only if** adding it lowers the mean log loss over folds 1–4
     **and** lowers it in at least 3 of the 4 folds. That's the same bar as steps 5 and 6.
   - Kept groups stay in; dropped groups are gone for good.
   - **Why groups and a fixed order:** testing every combination of 12 features would be 4,000+
     models on 4 folds, and the winner would be partly luck. A short, fixed path keeps the
     number of choices small.
   - **Amendment: a minimum gain of 0.001.** This was added after the first run, which kept %
     of games scoring first on a gain of +0.0001. That's a rounding error, and with only 4
     folds, pure noise clears "better than zero in 3 of 4" fairly often. 0.001 is about the
     smallest log-loss difference one season of data can tell apart from noise.
     - The amendment is applied to **every** group, and it's backed by steps 4 and 5: %
       scoring first is only partly skill, and it's a weaker copy of goal differential.
     - It doesn't use the fold 5 result.
     - It changes only that one decision: goal differential (+0.0014) still passes, and every
       other group was already rejected.
     - The original three-feature model is still scored on fold 5 below, so both versions
       stay visible.
   - **Recent form goes last.** Step 5 showed recent form is worse than the season figure
     *alone*. This tests whether it adds anything *on top of* it.
4. **Tune the penalty once,** on the final feature set, over a small grid of C values. Keep
   the default C = 1 unless another value beats it by more than one standard error (the
   simpler default wins ties).
5. **The intercept vs. switch check:** report the same final features with a plain intercept
   instead of the switch. This is for information only; the switch is used regardless,
   because it's the correct treatment of neutral sites.
6. **Final evaluation, once:** train on everything before fold 5, then predict fold 5 (Feb 21
   – Apr 11, including the tournaments). Report:
   - **log loss**, which is the main metric;
   - **AUC**;
   - **accuracy**;
   - **Brier score**;
   - **calibration**: do 60% predictions win about 60% of the time?

   Compare against two simple baselines: always predicting the home win rate, and a model
   using **win % only** ("the better record wins").

In [ ]:
%run ./02_choose_k.ipynb

In [ ]:
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.metrics import brier_score_loss, roc_auc_score

assert K == 5
data = rows.reset_index(drop=True)
y = data["home_win"].to_numpy()
splits = list(TimeSeriesSplit(n_splits=K).split(data))
SELECT, CONFIRM = splits[:4], splits[4]
print(f"{len(data)} games; {int((data['home_ice'] == 0).sum())} at neutral sites")

## The model

`home_ice` goes in unscaled as the switch. The stat features are scaled without centering.
There's no separate intercept.

In [ ]:
def make_model(features, C=1.0, switch=True):
    if switch:
        prep = ColumnTransformer([
            ("stats", StandardScaler(with_mean=False), features),  # 0 stays "evenly matched"
            ("home", "passthrough", ["home_ice"]),                   # the home ice switch
        ])
        return make_pipeline(prep, LogisticRegression(C=C, fit_intercept=False, max_iter=1000))
    prep = ColumnTransformer([("stats", StandardScaler(with_mean=False), features)])
    return make_pipeline(prep, LogisticRegression(C=C, max_iter=1000))  # plain intercept


def predict(features, train, test, **kw):
    m = make_model(features, **kw).fit(data.iloc[train], y[train])
    return m.predict_proba(data.iloc[test])[:, 1]


def fold_losses(features, folds, **kw):
    return np.array([log_loss(y[te], predict(features, tr, te, **kw), labels=[0, 1]) for tr, te in folds])


# Check the switch: a neutral-site game between identical teams must come out exactly 50/50
_m = make_model(["diff_close_shot_share_season"]).fit(data.iloc[SELECT[0][0]], y[SELECT[0][0]])
_even = pd.DataFrame({"diff_close_shot_share_season": [0.0, 0.0], "home_ice": [0, 1]})
_p = _m.predict_proba(_even)[:, 1]
assert abs(_p[0] - 0.5) < 1e-12, "neutral-site, evenly matched game should be 50/50"
print(f"Evenly matched: neutral site {_p[0]:.1%}, home game {_p[1]:.1%} (fold 1 training data)")

## Forward selection on folds 1–4

The fixed order follows step 5's ranking, with flagged and marginal features near the end
and recent form last.

In [ ]:
START = ("close-game shot share", ["diff_close_shot_share_season"])
GROUPS = [
    ("goal differential (no empty-netters)", ["diff_goal_diff_pg_no_en_season"]),
    ("shots for per game", ["diff_shots_for_pg_season"]),
    ("special teams (index + PP%)", ["diff_st_index_season", "diff_pp_pct_season"]),
    ("% of games scoring first", ["diff_scored_first_pct_season"]),
    ("streak", ["diff_streak"]),
    ("team save %", ["diff_save_pct_season"]),
    ("strength of schedule", ["diff_sos"]),
    ("PP chances per game (flagged in step 5)", ["diff_pp_chances_pg_season"]),
    ("recent form (ewm shot share + goal diff)", ["diff_shot_share_ewm", "diff_goal_diff_pg_no_en_ewm"]),
]

MIN_GAIN = 0.001  # amendment: smallest gain one season can separate from noise

base_const = np.array([log_loss(y[te], np.full(len(te), y[tr].mean()), labels=[0, 1]) for tr, te in SELECT])
current = list(START[1])
cur_loss = fold_losses(current, SELECT)
log = [{"step": "baseline (home win rate)", "log loss": base_const.mean(), "gain": np.nan,
        "folds improved": np.nan, "kept": None},
       {"step": f"start: {START[0]}", "log loss": cur_loss.mean(), "gain": base_const.mean() - cur_loss.mean(),
        "folds improved": int((cur_loss < base_const).sum()), "kept": True}]
for name, feats in GROUPS:
    trial = fold_losses(current + feats, SELECT)
    gain = cur_loss - trial
    keep = gain.mean() >= MIN_GAIN and (gain > 0).sum() >= 3
    log.append({"step": f"+ {name}", "log loss": trial.mean(), "gain": gain.mean(),
                "folds improved": int((gain > 0).sum()), "kept": keep})
    if keep:
        current, cur_loss = current + feats, trial

FINAL = current
pd.DataFrame(log).set_index("step").style.format({"log loss": "{:.4f}", "gain": "{:+.4f}", "folds improved": "{:.0f}"})

In [ ]:
print("Final feature set:")
for f in FINAL:
    print(f"  {f.removeprefix('diff_')}")
print("  + home ice switch")

## Tune the penalty once

A small grid around the default. **Lower C** = a stronger pull of the coefficients toward
zero. C = 1 stays unless another value is better by more than one standard error over the
4 folds.

In [ ]:
grid = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
by_c = {c: fold_losses(FINAL, SELECT, C=c) for c in grid}
tune = pd.DataFrame({c: {"mean log loss": v.mean()} for c, v in by_c.items()}).T
best_c = tune["mean log loss"].idxmin()
d = by_c[1.0] - by_c[best_c]
se = d.std(ddof=1) / np.sqrt(len(d))
C_FINAL = best_c if d.mean() > se else 1.0
print(f"Best C = {best_c}: better than C = 1 by {d.mean():+.4f} (SE {se:.4f}) -> using C = {C_FINAL}")
tune.style.format("{:.4f}")

## Switch vs. plain intercept (information only)

Same features and C. The switch is used either way, because it's the correct treatment of
neutral sites. This shows what it costs or gains on folds 1–4.

In [ ]:
sw = fold_losses(FINAL, SELECT, C=C_FINAL, switch=True)
ic = fold_losses(FINAL, SELECT, C=C_FINAL, switch=False)
print(f"Folds 1-4 log loss: switch {sw.mean():.4f}, plain intercept {ic.mean():.4f} "
      f"(difference {ic.mean() - sw.mean():+.4f}; only {int((data.iloc[np.concatenate([t for _, t in SELECT])]['home_ice'] == 0).sum())} neutral-site games in these test folds)")

## How the model reads

Coefficients of the final model fit on all games before fold 5:
- Each stat coefficient is the change in log-odds when the home-minus-away gap grows by one
  standard deviation.
- The home ice row, converted to a probability, is the evenly matched home team's win
  chance.

In [ ]:
train_all = np.arange(CONFIRM[1][0])
final_model = make_model(FINAL, C=C_FINAL).fit(data.iloc[train_all], y[train_all])
coef = final_model[-1].coef_[0]
names = [f.removeprefix("diff_") for f in FINAL] + ["home ice switch"]
coefs = pd.DataFrame({"coefficient": coef, "odds × per SD": np.exp(coef)}, index=names)
coefs.loc["home ice switch", "odds × per SD"] = np.nan
print(f"Evenly matched home team: {1 / (1 + np.exp(-coef[-1])):.1%}")
coefs.style.format({"coefficient": "{:+.3f}", "odds × per SD": "{:.2f}"}, na_rep="")

## Final evaluation on fold 5 (once)

Trained on every game before Feb 21. Fold 5 is the last 159 games of the season, including
the conference and NCAA tournaments. Nothing is changed after this cell.

In [ ]:
tr, te = CONFIRM
yt = y[te]
p_model = predict(FINAL, tr, te, C=C_FINAL)
p_base = np.full(len(te), y[tr].mean())
p_wins = predict(["diff_win_pct_season"], tr, te)
# The first run's model (before the minimum-gain amendment), kept for the record
V1 = ["diff_close_shot_share_season", "diff_goal_diff_pg_no_en_season", "diff_scored_first_pct_season"]
p_v1 = predict(V1, tr, te, C=C_FINAL)


def metrics(p):
    return {"log loss": log_loss(yt, p, labels=[0, 1]), "Brier": brier_score_loss(yt, p),
            "AUC": roc_auc_score(yt, p) if len(set(p)) > 1 else 0.5,
            "accuracy": ((p >= 0.5) == yt).mean()}


final = pd.DataFrame({
    "baseline: home win rate": metrics(p_base),
    "baseline: win % only": metrics(p_wins),
    "first run: + % scoring first": metrics(p_v1),
    "final model": metrics(p_model),
}).T
final.style.format({"log loss": "{:.4f}", "Brier": "{:.4f}", "AUC": "{:.3f}", "accuracy": "{:.1%}"})

### Calibration

Out-of-fold predictions from all five folds pooled (each game predicted by a model that
never saw it), grouped into probability bins. **A well-calibrated model sits on the
diagonal:** games given 60% are won about 60% of the time.

In [ ]:
oof = np.full(len(data), np.nan)
for trn, tst in splits:
    oof[tst] = predict(FINAL, trn, tst, C=C_FINAL)
mask = ~np.isnan(oof)
cal = pd.DataFrame({"p": oof[mask], "win": y[mask]})
cal["bin"] = pd.cut(cal["p"], [0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1])
cal_t = cal.groupby("bin", observed=True).agg(games=("win", "size"), predicted=("p", "mean"), actual=("win", "mean"))

BLUE, MUTED, GRID, INK = "#2a78d6", "#6b6a66", "#e1e0d9", "#1f1f1d"
fig, ax = plt.subplots(figsize=(5, 4.6))
ax.plot([0, 1], [0, 1], color=MUTED, lw=0.8, ls="--")
ax.plot(cal_t["predicted"], cal_t["actual"], "-o", color=BLUE, lw=2, ms=7)
for _, r in cal_t.iterrows():
    ax.annotate(f"{int(r['games'])} g", (r["predicted"], r["actual"]), textcoords="offset points",
                xytext=(6, -12), fontsize=8, color=MUTED)
ax.set_xlim(0.1, 0.95); ax.set_ylim(0.1, 0.95)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlabel("predicted home win probability", color=INK)
ax.set_ylabel("actual home win rate", color=INK)
ax.set_title("Calibration, out-of-fold (folds 1-5)", loc="left", fontsize=11)
ax.grid(color=GRID, lw=0.6)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()
print(f"{int(mask.sum())} games predicted out-of-fold; log loss {log_loss(y[mask], oof[mask]):.4f}")
cal_t.style.format({"predicted": "{:.1%}", "actual": "{:.1%}"})

## Conclusions

**1. The final feature set: two features plus the home ice switch.**

| Feature | Why it's in | Weight (odds × per SD) |
|---|---|---|
| Close-game shot share (season) | The strongest single predictor (step 5) | 2.07 |
| Goal differential, no empty-net goals (season) | Adds on top of shots: +0.0014, better in 4 of 4 folds | 1.22 |
| Home ice switch | An evenly matched home team wins 53.9%; 50/50 at neutral sites by rule | — |

**2. % of games scoring first is out (minimum-gain amendment).**
- The first run kept it on a gain of +0.0001, which is effectively zero.
- With the 0.001 minimum (see rule 3) it's rejected. Steps 4 and 5 already showed it's
  only partly skill and a weaker copy of goal differential.
- **The amendment changes nothing else.** Every other group fails as before, either for a
  negative gain or for improving fewer than 3 of 4 folds.

**3. Everything else failed to add anything on top:**
- shots for per game;
- special teams (the index and PP%);
- streak;
- team save %;
- PP chances per game;
- **recent form,** which made the model *worse* (−0.0042, 1 of 4 folds).

Once a team's season figures are known, its last few games add noise, not information.

**Strength of schedule** cleared the minimum gain (+0.0016), but improved only 2 of 4 folds,
so it fails the consistency bar. It's the first candidate to revisit with more seasons of
data.

**4. The penalty stays at the default (C = 1).** The best alternative was better by 0.0002,
well inside one SE. Stronger penalties hurt.

**5. The home ice switch costs nothing and is correct.** The two versions differ by 0.0005 in
log loss on folds 1–4, with only 30 neutral-site games there. The switch gives evenly
matched teams exactly 50/50 at a neutral site.

**6. On the held-back fold 5 (tournament season):**

| | Log loss | Brier | AUC | Accuracy |
|---|---|---|---|---|
| Home win rate | 0.675 | 0.241 | 0.500 | 64.8% |
| Win % only ("better record wins") | 0.639 | 0.224 | 0.615 | 64.2% |
| First run: + % scoring first | 0.626 | 0.218 | 0.642 | 67.9% |
| **Final model** | **0.627** | **0.219** | **0.639** | **66.7%** |

- **The two-feature model beats both baselines clearly.**
- **It's 0.0007 behind the first run's three-feature model on fold 5.** That's the size of
  noise on 159 games (about 2 games' worth of predictions changing sides), the same size
  the minimum-gain rule says can't be trusted. The simpler model is kept.
- **Accuracy is the weakest measure here.** Home teams won 64.8% of fold 5, so "always pick
  home" was already hard to beat on accuracy. Log loss and Brier score judge the
  probabilities themselves, which is what matters.

**7. Calibration is good in the middle and a little cautious at the extremes.**
- Across 795 out-of-fold predictions, the bins from 30% to 50% and from 60% to 80% agree
  within 1 point.
- The 50–60% bin under-predicts: 55% predicted against 63% actual, on 179 games.
- The extremes lean cautious: under 30% predicted 25% (actual 19%), and over 80% predicted
  84% (actual 94%, 36 games). Strong favorites win a bit more often than the model says.
- **For the model:** recheck on 2026-27 before adding any correction. With this few games at
  the extremes, the gap may not hold.

**8. What this means for building the model:**
- **Inputs:**
  - close-game shot share and goal differential without empty-net goals, both season to
    date and home minus away;
  - the home ice switch;
  - regularized logistic regression, C = 1, with the stat features scaled but not
    centered.
- **Training:** all of 2025-26 (957 games after the warm-up). Every feature uses only games
  before each game's date.
- **Test once on 2026-27,** which hasn't been touched at any point in this EDA. Expect AUC
  around 0.64–0.71 and log loss around 0.62–0.63: honest probabilities, not certainty.
- **Early season:**
  - Every feature is season to date, and the warm-up rule (3 games each) still applies, so
    in October the model leans mostly on the home edge.
  - A carry-over from the previous season (for example, a preseason Elo) is the natural next
    improvement. It can't be tested on 2025-26 games.